# Lab 05 — Active Learning: Designing and Evaluating Acquisition Policies

In previous labs we examined individual acquisition functions. Here we ask whether **combining** signals changes which messages are annotated and how efficiently a classifier learns.

We will compare individual methods with sequential selection, rank aggregation, and budget allocation. This is a retrospective simulation: BANKING77 labels are already available, but a simulated oracle reveals pool labels only after a policy selects examples. Evaluation examples remain outside the acquisition pool.

In [ ]:
from utils.data import build_representation
from utils.setup import prepare_dataset_experiment, run_experiments
from utils.analysis import plot_learning_curves, plot_final_performance, summarize_policies

## 1. Combining Acquisition Strategies

We have explored several ways to select examples for annotation. But instead of relying on a single acquisition signal, we can combine different strategies.

In this experiment, we compare three ways of combining acquisition methods:

- **Sequential selection:** Apply acquisition methods in stages.
- **Rank aggregation:** Combine the rankings produced by different methods.
- **Budget allocation:** Divide the annotation batch between strategies.

We will investigate whether these combinations improve learning compared with simpler acquisition policies.

In [ ]:
# Shared configuration for Experiment 1
EXPERIMENT = {
    # Data
    "dataset": "banking77",
    "representation": "tfidf",
    ##"train_size": 1000,           # Use 1,000 training candidates
    "split_seed": 42,
    "validation_fraction": 0.2,

    # Model and acquisition
    "model_type": "logistic_regression", # options: logistic_regression or multinomial_nb
    "initial_size": 90,
    "initial_method": "stratified_oracle",
    "budget": 200,
    "batch_size": 20,
    "seeds": [42, 43, 44, 45, 46],
}

# Use one run for quick exploration.
# Use several seeds to examine variability.
SEEDS = EXPERIMENT["seeds"]

print(f"Initial labeled examples: {EXPERIMENT['initial_size']}")
print(f"Additional annotation budget: {EXPERIMENT['budget']}")
print(f"Batch size: {EXPERIMENT['batch_size']}")
print(f"Acquisition rounds: {EXPERIMENT['budget'] // EXPERIMENT['batch_size']}")
print(f"Number of repetitions: {len(SEEDS)}")

In [ ]:
# Each dictionary is a readable policy configuration. Rank ties use pool order.
policies = {
    "Random": {"type": "random"},
    "Uncertainty": {"type": "uncertainty", "method": "least_confidence"},
    "Diversity": {"type": "diversity"},
    "Uncertainty → clustering": {
        "type": "sequential", "first": "uncertainty",
        "candidate_size": 100, "second": "clustering",
    },
    "Clustering → uncertainty": {
        "type": "cluster_then_uncertainty" # conceptually, this is sequential first: clustering, second: uncertainty
                                           # but required some special treatment so hence the specific type
    },
    "Rank: uncertainty + novelty": {
        "type": "rank_aggregation", "strategies": ["uncertainty", "novelty"],
        "weights": [0.5, 0.5],
    },
    "Budget 25/75": {
        "type": "budget_split", "strategies": ["uncertainty", "diversity"],
        "weights": [0.25, 0.75],
    },
    "Budget 50/50": {
        "type": "budget_split", "strategies": ["uncertainty", "diversity"],
        "weights": [0.5, 0.5],
    },
    "Budget 75/25": {
        "type": "budget_split", "strategies": ["uncertainty", "diversity"],
        "weights": [0.75, 0.25],
    },
    "Budget 75/25 (R)": {
        "type": "budget_split", "strategies": ["uncertainty", "random"],
        "weights": [0.75, 0.25],
    },    
}

In [ ]:
# Run all policies across the configured seeds.
results = run_experiments(EXPERIMENT, policies=policies)

# Index results by policy for convenient comparisons.
results_by_policy = {
    name: [r for r in results
           if r["configuration"]["policy_name"] == name]
    for name in policies
}


### 1.1 Single vs. Combined Acquisition Strategies

We begin by comparing random sampling, uncertainty, diversity, and a sequential combination of uncertainty and clustering.

The combined policy first identifies the 100 most uncertain examples, then selects 20 cluster representatives.

**Question:** Does combining uncertainty with clustering improve learning compared with the individual strategies?

In [ ]:
selection = [
    "Random",
    "Uncertainty",
    "Diversity",
    "Uncertainty → clustering",
]

comparison = [
    r for name in selection
    for r in results_by_policy[name]
]

plot_learning_curves(comparison, metric="macro_f1")

### 1.2 Does Acquisition Order Matter?

We compare two sequential acquisition policies:

- **Uncertainty → clustering:** Select 100 uncertain candidates, then choose 20 cluster representatives.
- **Clustering → uncertainty:** Cluster the entire pool into 20 groups, then select the most uncertain example from each group.

Both acquire 20 labels per round, but the order determines which examples can be selected.

**Question:** Does reversing the order change learning performance?


In [ ]:
selection = [
    "Uncertainty → clustering",
    "Clustering → uncertainty",
]

comparison = [
    r for name in selection
    for r in results_by_policy[name]
]

plot_learning_curves(comparison, metric="macro_f1")


### 1.3 Rank Aggregation

Instead of selecting examples in stages, we can combine the rankings produced by different acquisition signals.

Here, uncertainty and novelty receive equal weight. Examples with the best combined ranks are selected.

**Question:** Does balancing uncertainty and novelty improve learning compared with uncertainty alone?


In [ ]:
selection = [
    "Uncertainty",
    "Rank: uncertainty + novelty",
]

comparison = [
    r for name in selection
    for r in results_by_policy[name]
]

plot_learning_curves(comparison, metric="macro_f1")


### 1.4 Budget Allocation

Another approach is to divide each acquisition batch between different strategies.

We compare three allocations of uncertainty and diversity:

- **25/75:** 25% uncertainty, 75% diversity.
- **50/50:** Equal allocation.
- **75/25:** 75% uncertainty, 25% diversity.

With a batch size of 20, each policy still acquires the same number of examples per round.

**Question:** How does the allocation between uncertainty and diversity affect learning performance?


In [ ]:
selection = [
    "Uncertainty",
    "Diversity",
    "Budget 25/75",
    "Budget 50/50",
    "Budget 75/25",
    #"Budget 75/25 (R)",
]

comparison = [
    r for name in selection
    for r in results_by_policy[name]
]

plot_learning_curves(comparison, metric="macro_f1")


### 1.5 Comparing Acquisition Policies

We have tested different ways of combining acquisition signals under the same experimental conditions.

Let's compare all policies at the full annotation budget.

**Questions**

- Which policies achieved the highest final performance?
- Were their advantages consistent across repeated runs?
- Did the combined policies justify their additional complexity?

Remember that these results apply to our particular dataset, representation, classifier, and annotation budget.

In [ ]:
plot_final_performance(
    results,
    metric="macro_f1",
    budget=EXPERIMENT["budget"],
)

display(
    summarize_policies(results)
    .sort_values("macro_f1_mean", ascending=False)
)

## 2. Fixed vs. Adaptive Acquisition Policies

So far, we have used the same acquisition policy throughout the active-learning process. But the examples that are useful early in training may not be the same as those that are useful later.

We will compare three approaches:

- **Fixed:** Use the same acquisition strategy in every round.
- **Scheduled:** Switch between strategies according to a predefined sequence.
- **Feedback-adaptive:** Use the loss on newly labeled examples to decide which strategy to use next.

**Question:** Does changing the acquisition strategy over time improve learning compared with using a fixed policy?

### 2.1 Configuring the Policies

We compare four policies:

| Policy | Acquisition behavior |
|---|---|
| **Fixed uncertainty** | Select the most uncertain examples in every round. |
| **Fixed diversity** | Use farthest-first sampling in every round. |
| **Scheduled** | Follow a predefined sequence of diversity and uncertainty rounds. |
| **Adaptive** | Use feedback from newly labeled batches to switch between diversity and uncertainty. |

For the adaptive policy, we use the approach discussed in the lecture, adapted from Jung et al.:

1. Measure the loss on newly labeled examples using the model **before retraining**.
2. Normalize that loss relative to recent batches.
3. Update the trade-off parameter β.
4. Use diversity when β ≤ 0.5 and uncertainty when β > 0.5.

All policies use the same model, representation, starting conditions, and annotation budget.


In [ ]:
from utils.analysis import plot_policy_dynamics

# Experiment 2 uses the same data, initialization, budget, and seeds.
EXPERIMENT_2 = {**EXPERIMENT}


In [ ]:
policies_2 = {
    # FIXED POLICIES: Use the same acquisition method in every round.

    "Fixed uncertainty": {
        "type": "uncertainty",
        "method": "least_confidence",  # Select examples with the lowest maximum class probability.
    },

    "Fixed diversity": {
        "type": "diversity",  # Greedy farthest-first: select examples distant from those already selected.
    },

    # SCHEDULED POLICY: Follow a predefined sequence of acquisition methods.

    "Scheduled": {
        "type": "scheduled",
        # Each entry specifies the method used for one annotation batch.
        # Start with exploration, then increasingly favor uncertainty.
        "schedule": [
            "diversity", "diversity",        # Rounds 1–2: Explore
            "uncertainty", "uncertainty",    # Rounds 3–4: Exploit
            "diversity",                   # Round 5: Explore again
            "uncertainty", "uncertainty",    # Rounds 6–7: Exploit
            "uncertainty", "uncertainty", "uncertainty",  # Rounds 8–10
        ],
        "repeat": True,  # Restart the schedule if there are more than 10 rounds.
    },

    # FEEDBACK-ADAPTIVE POLICY: Change methods based on losses observed
    # on newly labeled batches (using the model before retraining).

    "Adaptive": {
        "type": "adaptive",
        "explore": "diversity",       # Explore the feature space.
        "exploit": "uncertainty",     # Focus on uncertain predictions.
        "initial_beta": 0.5,         # Initial exploration/exploitation trade-off.
        "lambda": 0.9,               # Multiplicative decay factor in the beta update.
        "window": 3,                 # Compare the current loss with the last 3 batch losses.
        "threshold": 0.5,            # beta <= 0.5: diversity; beta > 0.5: uncertainty.
        "epsilon": 0.1,              # Keep beta within [0.1, 0.9].
    },
}

In [ ]:
results_2 = run_experiments(EXPERIMENT_2, policies_2)

### 2.2 Comparing Learning Performance

Let's compare how the four policies perform as the annotation budget increases.

The scheduled and adaptive policies can change their acquisition method during learning, while the fixed policies cannot.

Does this flexibility lead to better classification performance?


In [ ]:
plot_learning_curves(results_2, metric="macro_f1")

In [ ]:
plot_final_performance(
    results_2,
    metric="macro_f1",
    budget=EXPERIMENT_2["budget"],
)

**Discussion**

- Did the scheduled or adaptive policy outperform fixed uncertainty sampling?
- Did their relative performance change throughout the acquisition process?
- Does a more flexible acquisition policy necessarily improve learning?

### 2.3 How Did the Adaptive Policy Behave?

The adaptive policy updates β after each newly labeled batch.

- **β <= 0.5:** Explore using diversity.
- **β > 0.5:** Exploit using uncertainty.

Let's examine one experimental run to see how feedback affected its decisions.

Remember that the policy is not guaranteed to switch regularly. Its behavior depends on the losses observed during learning.


In [ ]:
from utils.analysis import plot_policy_dynamics

# Select one run to inspect
adaptive_run = next(
    r for r in results_2
    if r["configuration"]["policy_name"] == "Adaptive"
    and r["configuration"]["random_state"] == EXPERIMENT_2["seeds"][0]
)

plot_policy_dynamics(adaptive_run)

**Discussion**

- How often did the adaptive policy explore versus exploit?
- Did feedback lead to frequent switching, or did one strategy dominate?
- How might this behavior help explain the learning curves?

**Important:** Adaptive acquisition allows the strategy to respond to newly acquired labels, but adaptation does not guarantee better learning. Its effectiveness depends on the feedback signal, update rule, and dataset.

## 3. Managing the Annotation Budget

So far, we have compared acquisition policies using a fixed initial labeled set, batch size, and annotation budget.

But these are also important design decisions. How many examples should we label before training the first model? How frequently should we update it? And when should we stop requesting more labels?

In this experiment, we investigate three aspects of budget management:

1. **Initialization:** How should we select the first labeled examples?
2. **Batch size:** How often should we acquire labels and retrain?
3. **Stopping:** When do we have enough labeled data?

Unlike previous experiments, our **total annotation budget includes both initialization and subsequent acquisitions**. Different configurations may start with different numbers of labels or stop before exhausting the budget.

In [ ]:
from utils.setup import run_conditions
from utils.analysis import (
    summarize_conditions,
    plot_initialization_cost,
    plot_stopping_outcomes,
    plot_stopping_diagnostics,
    summarize_stopping,
)

# Reuse the dataset settings; the total budget includes initialization.
EXPERIMENT_3 = {
    key: value for key, value in EXPERIMENT.items()
    if key not in ("budget", "initial_size", "initial_method")
}
EXPERIMENT_3.update({
    "initial_method": "random",
    "min_per_class": 2,
    "total_budget": 200,
})

# Use the same acquisition strategy after initialization.
uncertainty = {"type": "uncertainty", "method": "least_confidence"}

### 3.1 How Should We Initialize the Model?

Before the first classifier can be trained, we need an initial labeled dataset.

We compare three initialization strategies:

| Strategy | How the next example is selected |
|---|---|
| **Random** | Select an example randomly. |
| **K-means++** | Favor examples distant from previously selected examples. |
| **Nearest-Neighbor Coverage (NNC)** | Select the example that best improves coverage of the unlabeled pool. |

Each strategy continues until we have observed **at least two labeled examples per class**.

Because the strategies discover classes at different rates, they may require different numbers of annotations. All annotations count toward the same total budget.

**Questions:**
- Which strategy discovers all classes with fewer annotations?
- Does more efficient initialization lead to better subsequent learning?

*Note: We use the known set of dataset classes to determine when initialization is complete. The selection methods themselves do not use the hidden labels.*


In [ ]:
# Compare how we select the initial labeled examples.

initialization_conditions = {
    "Random": {
        "initial_method": "random",
    },
    "K-means++": {
        "initial_method": "kmeans++",
    },
    "NNC": {
        "initial_method": "nnc",
        "candidate_size": 600,  # Sample 600 candidates per seed for NNC initialization.
    },
}

results_3a = run_conditions(
    EXPERIMENT_3,
    initialization_conditions,
    policy=uncertainty,
)

In [ ]:
# How many annotations were needed to discover 2 examples per class?
plot_initialization_cost(results_3a)

_, initialization_summary = summarize_conditions(results_3a)
display(initialization_summary.round(3))



#### Does Initialization Affect Subsequent Learning?

A method that discovers the classes with fewer annotations leaves more of the budget for active acquisition.

Let's compare how the resulting classifiers improve as additional labels are acquired.

The horizontal axis now shows the **total number of labeled examples**, including initialization.


In [ ]:
plot_learning_curves(
    results_3a,
    metric="macro_f1",
    x_axis="training_size",
    aggregate=True
)


### 3.2 How Often Should We Update the Model?

In batch active learning, we acquire several examples before retraining.

A smaller batch allows the model to incorporate feedback more frequently. A larger batch requires fewer training rounds, but selects more examples using the same model.

We compare three batch sizes:

- **5:** Frequent model updates.
- **20:** Moderate update frequency.
- **50:** Fewer, larger acquisition rounds.

All conditions use the same initialization method, acquisition strategy, and maximum annotation budget.

**Question:** Does updating the model more frequently improve label efficiency?


In [ ]:
# Change only how many examples are acquired before retraining.
batch_conditions = {
    "Batch 5": {"batch_size": 5},
    "Batch 20": {"batch_size": 20},
    "Batch 50": {"batch_size": 50},
}

results_3b = run_conditions(
    EXPERIMENT_3,
    batch_conditions,
    policy=uncertainty,
)

In [ ]:
plot_learning_curves(
    results_3b,
    metric="macro_f1",
    x_axis="training_size",
    aggregate=True
)

_, batch_summary = summarize_conditions(results_3b)
display(batch_summary.round(3))


**Discussion**

- Which batch size achieved the best learning performance?
- Did smaller batches improve label efficiency, particularly early in learning?
- What is the trade-off between learning performance and the number of model updates?

Remember that smaller batches require more training rounds, even when the total number of annotations is the same.

### 3.3 When Should We Stop Acquiring Labels?

Until now, our experiments have continued until the annotation budget was exhausted.

But additional labels may provide diminishing improvements. We can introduce stopping criteria to decide when further annotation is no longer necessary.

We compare three stopping conditions:

| Criterion | Stop when... |
|---|---|
| **Maximum budget** | The total annotation budget has been exhausted. |
| **Target performance** | The model reaches a specified performance on the validation set. |
| **Prediction stability** | Consecutive models make sufficiently similar predictions on a fixed reference set. |

For prediction stability, we use the approach discussed in class, based on Bloodgood and Vijay-Shanker: calculate Cohen's κ between consecutive model predictions and stop when the average over the last three comparisons reaches 0.99.

The maximum budget always remains a limit. If another stopping criterion is reached first, acquisition ends early.

**Question:** Can we reduce the number of annotations without substantially reducing model performance?

In [ ]:
stopping_conditions = {
    # Continue until the total annotation budget is exhausted.
    "Maximum budget": {},

    # Stop once validation macro-F1 reaches 0.70.
    "Target macro-F1": {
        "stopping": {
            "target_performance": {
                "metric": "macro_f1",
                "threshold": 0.85,
            },
        },
    },

    # Stop when predictions become sufficiently stable:
    # Mean Cohen's kappa >= 0.99 over the last 3 comparisons.
    "Prediction stability": {
        "stopping": {
            "prediction_stability": {
                "metric": "cohen_kappa",
                "threshold": 0.90,
                "window": 3,
            },
        },
    },
}


In [ ]:
results_3c = run_conditions(
    EXPERIMENT_3,
    stopping_conditions,
    policy=uncertainty,
)

In [ ]:
# What happened: when did each run stop, and what triggered it?
plot_stopping_outcomes(results_3c)

# Why: did the configured thresholds activate?
plot_stopping_diagnostics(results_3c, seed=SEEDS[0])

# At what cost: labels consumed, labels saved, and final performance.
display(summarize_stopping(results_3c).round(3))

### 3.4 Comparing Budget Management Decisions

Our experiments show that annotation efficiency depends on more than choosing an acquisition strategy.

- **Initialization** determines how we spend our first annotations.
- **Batch size** determines how frequently the model can use new feedback.
- **Stopping criteria** determine when further annotation is no longer necessary.

These decisions interact with the dataset, representation, model, and acquisition policy.

**Final question:** If you were designing an active-learning system with a limited annotation budget, which of these decisions would you prioritize, and why?


## 4. Does Our Acquisition Policy Generalize?

In the previous experiments, we compared acquisition strategies, adaptive policies, and annotation-budget decisions. Most comparisons used the same dataset, representation, and classifier.

But a policy that performs well in one setting may behave differently in another.

In this final experiment, we investigate two questions:

1. **Representation:** Does changing how we represent the data affect the benefit of active learning?
2. **Dataset:** Does the same acquisition strategy work equally well on a different classification task?

We will compare **random sampling** and **uncertainty sampling** under different conditions.

Our goal is not to identify a universally best policy, but to understand why acquisition strategies must be evaluated in the context where they will be used.


### 4.1 Does the Representation Matter?

An acquisition policy relies on the information available to the classifier.

Changing the text representation can affect both what the model learns and which examples it considers uncertain.

We compare two representations on BANKING77:

- **TF-IDF:** Represents messages using word importance.
- **GloVe:** Represents messages using pretrained word embeddings.

For each representation, we run the same two acquisition policies:

- **Random:** Select examples without using model predictions.
- **Uncertainty:** Select examples with the lowest maximum predicted class probability.

We keep the classifier, initialization, and annotation budget fixed, using the same initial examples across representations for each seed.

**Question:** Does uncertainty sampling provide the same advantage over random sampling with both representations?


In [ ]:
from utils.setup import run_dataset_experiment
from utils.analysis import (
    plot_setting_learning_curves,
    plot_dataset_learning_curves,
    plot_policy_advantages,
)

policies_4 = {
    "Random": {"type": "random"},
    "Uncertainty": {
        "type": "uncertainty",
        "method": "least_confidence",
    }, 
}

BANKING = {
    "dataset": "banking77",
    "representations": [
        {"method": "tfidf"},
        {"method": "embeddings"},  # Existing GloVe embeddings
    ],
    "classifier": "logistic_regression",
    "initial_per_class": 10,
    "total_budget": 200,
    "batch_size": 20,
    "seeds": [42, 43, 44, 45, 46],
    "split_seed": 42,
}

banking_results = run_dataset_experiment(BANKING, policies_4)


#### Comparing Learning Curves

Let's compare the learning curves for random and uncertainty sampling under the two representations.

Within each representation, the policies use the same initial labeled examples and total annotation budget.

When interpreting the results, consider both the overall classification performance and the difference between uncertainty and random sampling.


In [ ]:
plot_setting_learning_curves(banking_results)



**Discussion**

- Does one representation produce better classification performance?
- Does uncertainty sampling outperform random sampling with both representations?
- Does the representation change the size of that advantage?

Remember that changing the representation affects both the classifier's input and the information used for acquisition.


In [ ]:
MODELS_COMP = {
    "dataset": "banking77",
    "representations": [
        {"method": "tfidf"},
    ],
    "classifier": "multinomial_nb",
    "initial_per_class": 10,
    "total_budget": 200,
    "batch_size": 20,
    "seeds": [42, 43, 44, 45, 46],
    "split_seed": 42,
}

models_results = run_dataset_experiment(MODELS_COMP, policies_4)

In [ ]:
plot_setting_learning_curves(models_results)

## 4.2 Does the Result Transfer to Another Dataset?

So far, we have focused on banking intent classification. But acquisition policies may behave differently on other tasks.

We will now use **20 Newsgroups**, a dataset of discussion posts classified into 20 topics.

Unlike BANKING77, which contains short customer-service messages, 20 Newsgroups contains longer texts on a wider range of subjects.

We will use TF-IDF and logistic regression, comparing random and uncertainty sampling under the same experimental procedure.

**Question:** Does uncertainty sampling offer a similar advantage over random sampling on a different classification task?


In [ ]:
NEWSGROUPS = {
    "dataset": "20newsgroups",
    "max_classes": 5,
    "train_size": 3000,
    "eval_size": 500,
    "representations": [
        {
            "method": "tfidf",
            "max_features": 1000,
            "min_df": 2,
            "max_df": 0.95,
        },
    ],
    "classifier": "logistic_regression",
    "initial_per_class": 5,
    "total_budget": 700,
    "batch_size": 20,
    "seeds": [42, 43, 44, 45, 46],
    "split_seed": 42,
}

# policies_5 = {
#     **policies_4,
    
#     "Uncertainty → clustering": {
#         "type": "sequential", "first": "uncertainty",
#         "candidate_size": 100, "second": "clustering",
#     },
#     "Budget 75/25 (R)": {
#         "type": "budget_split", "strategies": ["uncertainty", "random"],
#         "weights": [0.75, 0.25],
#     }, 
# }

newsgroups_results = run_dataset_experiment(
    NEWSGROUPS,
    policies_4,
)

### Comparing Across Datasets

We now have results from two classification tasks.

Rather than comparing their absolute macro-F1 scores directly, we will examine how much uncertainty sampling improves over random sampling **within each task**.

The datasets differ in their number of classes, text characteristics, and classification difficulty, so a higher score on one dataset does not necessarily indicate a more effective acquisition policy.

Let's compare their learning curves side by side.


In [ ]:
banking_tfidf_results = [
    result for result in banking_results
    if result["configuration"]["representation"]["method"] == "tfidf"
]
plot_dataset_learning_curves(banking_tfidf_results, newsgroups_results)


**Discussion**

- Does uncertainty outperform random sampling on both datasets?
- Is the advantage similar throughout the annotation process?
- What differences between the tasks might explain the observed results?


In [ ]:
plot_policy_advantages(
    banking_results + newsgroups_results,
    baseline="Random",
    policy="Uncertainty",
)
